# Loading PDF


In [2]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)


## PyPDFLoader (most common technique)

In [8]:
print("PyPDFLoader:")

try:
    py_pdfLoader= PyPDFLoader("data/AYUSH_RESUME (1).pdf")
    pdfDoc= py_pdfLoader.load()
    print("Pdf is loaded successfully using PyPDFLoader")
    print(pdfDoc[0].page_content[:500])
    print(f"Metadata ", pdfDoc[0].metadata)  #  Print the first 500 characters of the first page 
except Exception as e:
    print(f"Error loading PDF with PyPDFLoader: {e}")

PyPDFLoader:
Pdf is loaded successfully using PyPDFLoader
Ayush Pandit
Raipur, Chhattisgarh, India - 490042
♂phone+91 9755260855 /envel⌢pepanditayush2703@gmail.com /linkedinLinkedin /githubGithub
Summary
Software Engineer focused on buildingscalable enterprise platforms and developer tooling. Experience
acrossNode.js, React, distributed systems, and real-time services, with strong emphasis onAPI
platforms, workflow orchestration, observability, and performance. Familiar withGenerative AI
fundamentals, Jupyter-based experimentation, and AI-assisted deve
Metadata  {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-04T11:05:38+00:00', 'author': '', 'keywords': '', 'moddate': '2026-03-04T11:05:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/AYUSH_RESUME (1).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


## PyMuPDF (fast and accurate)



In [9]:
print("\nPyMuPDFLoader:")
try:
    pymuPdfDoc= PyMuPDFLoader("data/AYUSH_RESUME (1).pdf").load()
    print("Pdf is loaded successfully using PyMuPDFLoader")
    print(pymuPdfDoc[0].page_content[:500])
    print(f"Metadata ", pymuPdfDoc[0].metadata)  #  Print
except Exception as e:
    print(f"Error loading PDF with PyMuPDFLoader: {e}")



PyMuPDFLoader:
Pdf is loaded successfully using PyMuPDFLoader
Ayush Pandit
Raipur, Chhattisgarh, India - 490042
 +91 9755260855
# panditayush2703@gmail.com
ï Linkedin
§Github
Summary
Software Engineer focused on building scalable enterprise platforms and developer tooling. Experience
across Node.js, React, distributed systems, and real-time services, with strong emphasis on API
platforms, workflow orchestration, observability, and performance. Familiar with Generative AI
fundamentals, Jupyter-based experimentation, and AI-assisted development tools, enabl
Metadata  {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-04T11:05:38+00:00', 'source': 'data/AYUSH_RESUME (1).pdf', 'file_path': 'data/AYUSH_RESUME (1).pdf', 'total_pages': 2, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-03-04T11:05:38+00:00', 'trapped': '', 'modDate': 'D:20260304110538Z', 'creationDate': 'D:20260304110538Z', 'page': 0}


## PDF loaders comparison

### PyPDFLoader 
  -Simple , reliable and good for most pdfs
  -Preserve Page numbers
  -Basic text extraction not best
  *** Use for standard text pdf

### PyMuPDFLoader   
  -Fast processing
  -Good Text Extraction
  -Image extraction support
  *** Use when speed is important 

## Handling PDF challenges 

-Store text in complex ways
-can have formatting issues
-May contained Scanned images(OCR required)
-Often have extraction artifacts 

In [13]:
## Example of raw PDF for cleanig 
raw_pdf= """Ayush Kumar
Data Scientist | Machine Learning Engineer



    The financical condition of the company is very good. The company has been growing rapidly in the last few years and has a strong financial position. The company has a strong balance sheet and a healthy cash flow. The company has a strong revenue growth and a strong profit margin. The company has a strong return on equity and a strong return on assets. The company has a strong dividend yield and a strong dividend payout ratio. The company has a strong stock price performance and a strong market capitalization. The company has a strong competitive position 
     
      
        and a strong market share. The company has a strong management team and a strong board of directors. The company has a strong corporate governance and a strong social responsibility. The company has a strong environmental responsibility and a strong sustainability. The company has a strong innovation and a strong research and development. The company has a strong customer satisfaction and a strong employee satisfaction. The company has a strong brand recognition and a strong reputation. The company has a strong growth potential and a strong future outlook.

"""

##Apply clean funciton

def clean_pdf_text(raw_pdf):
    # REmove excessivce whitespace
    cleanText= " ".join(raw_pdf.split())

    #Fix ligatures
    cleanText= cleanText.replace("ﬁ", "fi").replace("ﬂ", "fl")
    return cleanText

cleaned_pdf= clean_pdf_text(raw_pdf)
print(f"before cleaning: {raw_pdf}")
print(f"after cleaning: {cleaned_pdf[:100]}...")





before cleaning: Ayush Kumar
Data Scientist | Machine Learning Engineer



    The financical condition of the company is very good. The company has been growing rapidly in the last few years and has a strong financial position. The company has a strong balance sheet and a healthy cash flow. The company has a strong revenue growth and a strong profit margin. The company has a strong return on equity and a strong return on assets. The company has a strong dividend yield and a strong dividend payout ratio. The company has a strong stock price performance and a strong market capitalization. The company has a strong competitive position 


        and a strong market share. The company has a strong management team and a strong board of directors. The company has a strong corporate governance and a strong social responsibility. The company has a strong environmental responsibility and a strong sustainability. The company has a strong innovation and a strong research and development. The com

## A real processing pdf and chunking them


In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

class SmartPDFProcessor:
    """Advance PDF processor that can handle complex PDF structures and extract clean text for downstream tasks."""
    def __init__(self,chunk_size=1000,chunk_overlap=200):
        self.chunk_size= chunk_size
        self.chunk_overlap= chunk_overlap
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]

        )

    def clean_pdf_text(self, text):
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        text = re.sub(r'\s+\n', '\n', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    def process_pdf(self,pdf_path):

        try:
            loader=PyMuPDFLoader(pdf_path)
            pages= loader.load()
            print(f"Loaded PDF with {len(pages)} pages.")
        except Exception as e:
            print(f"Error loading PDF: {e}")
            return []
        
        all_chunks=[]

        for page_num,page in enumerate(pages):
            cleaned_text=self.clean_pdf_text(page.page_content)

            if len(cleaned_text)<50:
                print(f"Skipping page {page_num} due to insufficient content.")
                continue

        
            chunk=self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[
                    {**page.metadata,
                     "page": page_num+1,
                     "total_pages": len(pages),
                     "chunk_method": "recursive_character",
                     "char_count": len(cleaned_text),
                     "source": pdf_path,
                     "page_number": page_num, 
                     "document_id": hash(pdf_path)}
                     ]
            )
            all_chunks.extend(chunk)
            print(f"Processed page {page_num}: {len(chunk)} chunks created.")

        return all_chunks
    



In [26]:
processPDF= SmartPDFProcessor()
processPDF

In [28]:
smartchunks=processPDF.process_pdf("data/Employment Contract.pdf")
print(f"Total chunks created: {len(smartchunks)}")
for i,chunk in enumerate(smartchunks[:3]):
    print(f"\nChunk {i+1}:")
    print(f"Metadata:")
    for key, value in chunk.metadata.items():
        print(f"{key}: {value}")
    print(f"Content: {chunk.page_content[:200]}...")


Loaded PDF with 8 pages.
Processed page 0: 3 chunks created.
Processed page 1: 4 chunks created.
Processed page 2: 3 chunks created.
Processed page 3: 4 chunks created.
Processed page 4: 4 chunks created.
Processed page 5: 4 chunks created.
Processed page 6: 3 chunks created.
Processed page 7: 1 chunks created.
Total chunks created: 26

Chunk 1:
Metadata:
producer: www.ilovepdf.com
creator: Microsoft® Word 2016
creationdate: 2026-01-07T03:14:54+00:00
source: data/Employment Contract.pdf
file_path: data/Employment Contract.pdf
total_pages: 8
format: PDF 1.5
title: 
author: panditayush2703@outlook.com
subject: 
keywords: 
moddate: 2026-01-07T03:14:54+00:00
trapped: 
modDate: D:20260107031454Z
creationDate: D:20260107031454+00'00'
page: 1
chunk_method: recursive_character
char_count: 2056
page_number: 0
document_id: -2876020960348279233
Content: Employment Contract
This Comprehensive Employment Contract (the “Contract”) is made and entered into effective
as of November 29, 2025 (the “Effe